<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# PDF to Text — Utility Notebook

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~0h10 (run all cells)

</div>

## 📋 Overview

This **utility notebook** converts PDF and TXT files from `gs://bucket-lab05/pdfs/` into `.txt` files and uploads them to your personal folder `gs://bucket-lab05/{YOUR_NAME}/documents/`. PDFs are converted to text; TXT files are copied as-is. It is a prerequisite for Lab 05a — run it **before** starting the RAG lab.

> 💡 **Tip:** You only need to run this notebook once. After execution, your `.txt` documents will be available in Cloud Storage for the RAG corpus import.

---
## 0 · Setup

### 0.1 Install dependencies

In [ ]:
%pip install PyPDF2 google-cloud-storage -q

### 0.2 Imports

In [ ]:
from google.cloud import storage
import PyPDF2
import os

print("✅ Imports complete")

### 0.3 Configuration

In [ ]:
# ── Configuration ──
##############################  TODO  ##############################
BUCKET_NAME = "bucket-lab05"  # @param {type:"string"}
YOUR_NAME = "ABC"  # @param {type:"string"}
####################################################################

SOURCE_PREFIX = "pdfs"  # Shared folder with PDF files
DEST_PREFIX = f"{YOUR_NAME}/documents"  # Your personal output folder
LOCAL_FOLDER = "pdf_temp"

print(f"Source: gs://{BUCKET_NAME}/{SOURCE_PREFIX}/")
print(f"Destination: gs://{BUCKET_NAME}/{DEST_PREFIX}/")

---
## 1 · Helper Functions

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Extract all text from a PDF file using PyPDF2."""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        text = ""
        for page in reader.pages:
            text += page.extract_text()
    return text


def retrieve_source_files(bucket, prefix):
    """List all .pdf and .txt files under a given prefix in a GCS bucket."""
    files = [
        blob.name for blob in bucket.list_blobs(prefix=prefix)
        if blob.name.endswith(".pdf") or blob.name.endswith(".txt")
    ]
    print(f"Found {len(files)} file(s) in gs://{bucket.name}/{prefix}/")
    for f in files:
        print(f"  • {f}")
    return files


def download_from_gcs(bucket, blob_name, destination):
    """Download a file from GCS to a local path."""
    blob = bucket.blob(blob_name)
    blob.download_to_filename(destination)


def upload_to_gcs(bucket, local_path, blob_name):
    """Upload a local file to GCS."""
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(local_path)

---
## 2 · Run Pipeline

In [ ]:
def pdf_to_text_pipeline(bucket_name, source_prefix, dest_prefix, local_folder):
    """Download files from source prefix → convert PDFs to text / copy TXTs → upload to dest prefix."""
    os.makedirs(local_folder, exist_ok=True)

    client = storage.Client()
    bucket = client.bucket(bucket_name)

    source_files = retrieve_source_files(bucket, source_prefix)

    for source_file in source_files:
        try:
            basename = os.path.basename(source_file)
            local_path = os.path.join(local_folder, basename)
            download_from_gcs(bucket, source_file, local_path)

            # Build the .txt output name (replace only the final extension)
            name_without_ext, ext = os.path.splitext(basename)

            if ext.lower() == ".pdf":
                # Convert PDF → TXT
                text = extract_text_from_pdf(local_path)
                txt_name = f"{name_without_ext}.txt"
                local_txt = os.path.join(local_folder, txt_name)
                with open(local_txt, "w") as f:
                    f.write(text)
            elif ext.lower() == ".txt":
                # Already TXT — just copy as-is
                txt_name = basename
                local_txt = local_path
            else:
                print(f"⏭️ Skipping unsupported format: {basename}")
                continue

            dest_blob_name = f"{dest_prefix}/{txt_name}"
            upload_to_gcs(bucket, local_txt, dest_blob_name)
            print(f"✅ {source_file} → {dest_blob_name}")

        except Exception as e:
            print(f"❌ Error processing {source_file}: {e}")

    print("\n✅ Pipeline complete.")

In [ ]:
pdf_to_text_pipeline(BUCKET_NAME, SOURCE_PREFIX, DEST_PREFIX, LOCAL_FOLDER)

---
## Summary

This notebook converted your PDF documents to `.txt` format and uploaded them to `gs://bucket-lab05/{YOUR_NAME}/documents/`.

**Next step:** Open **Lab 05a — RAG Corpus & Retrieval** to build a RAG knowledge base from these documents.